# LoRA-corrected teacher eval: MBPP-TS (full 390-problem set)

Part of plan.md step 3.3 -- the second gating benchmark, run only against
the LoRA-corrected teacher (`sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora`). Not run against the
full-parameter corrected teacher: that checkpoint was trained before the
3.4 data-quality fix (truncated samples, `node_modules`/`.d.ts`
contamination), so its MBPP-TS score wouldn't be a fair data point --
we already know its HumanEval-TS result reflects the unfixed data.

Same suite/harness as `11b_teacher_corrected_lora_humaneval_ts.ipynb`,
same export-stripping fix (`bonsai.eval.register`), `batch_size=8`
(established: `batch_size=32` hangs on this MoE model's decode-bound
generation). 390 problems -- 2.45x HumanEval-TS's 159, so expect
proportionally longer: HumanEval-TS took ~2h23min for this model, so
budget roughly 5.5-6 hours.

Needs a rented GPU (1x 80GB) with the `eval` extra installed.

In [ ]:
import json

from bonsai.config import load_config
from bonsai.tracking import new_run, save_config_snapshot

config = load_config("../configs/qwen3-30b-a3b.toml")
model_repo_id = "sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora"
run_dir = new_run("teacher-corrected-lora-eval")
save_config_snapshot(run_dir, config)

print(f"model: {model_repo_id}")
print(f"run dir: {run_dir}")

In [ ]:
import os

os.environ["MODEL_REPO_ID"] = model_repo_id
os.environ["MODEL_REVISION"] = "main"
os.environ["RUN_DIR"] = str(run_dir)

In [ ]:
!stdbuf -o0 -e0 accelerate launch ../tools/bigcode_eval_main.py \
    --model "$MODEL_REPO_ID" --revision "$MODEL_REVISION" --precision bf16 \
    --tasks multiple-mbpp-ts \
    --max_length_generation 512 --batch_size 8 \
    --allow_code_execution --save_generations \
    --metric_output_path "$RUN_DIR/mbpp-ts.json" \
    --save_generations_path "$RUN_DIR/mbpp-ts-generations.json" \
    2>&1 | tee /tmp/live_mbpp_ts_lora_full.log

assert _exit_code == 0, "accelerate launch failed for multiple-mbpp-ts"  # noqa: F821 -- set by the ! shell magic above at runtime

In [ ]:
mbpp_ts = json.loads((run_dir / "mbpp-ts.json").read_text())
print("mbpp-ts (LoRA-corrected teacher, full 390-problem set):", mbpp_ts)

## Comparison

| | HumanEval-TS pass@1 | MBPP-TS pass@1 |
|---|---|---|
| base model (unpruned, uncorrected) | 0.6708 | 0.7333 |
| LoRA-corrected teacher | 0.5723 (0.5849 console.log-stripped) | see above |